# Topic 3 - The Context Window, Precisely

A context window is the maximum number of tokens a model can process in one request. This notebook turns the notes into small experiments so the limits become concrete.

## Learning objectives

By the end of this notebook, I should be able to:

- calculate how input and requested output share a context limit
- distinguish a hard overflow error from truncation or compaction
- explain why longer context increases attention work, latency, and memory use
- identify what may have been lost when a model appears to forget
- choose an explicit context-management policy for a request

## 1. Input and output share one budget

A practical constraint is:

`input tokens + maximum output tokens <= model context limit`

The output setting is a ceiling, not a promise. Tool results, retrieved documents, and conversation history also consume the request budget.

In [1]:
def context_budget(context_limit, input_tokens, requested_output):
    remaining = context_limit - input_tokens
    fits = requested_output <= remaining
    return {
        'remaining_output_capacity': max(0, remaining),
        'fits': fits,
        'overflow_tokens': max(0, input_tokens + requested_output - context_limit),
    }

context_budget(context_limit=16_000, input_tokens=12_000, requested_output=4_000)

{'remaining_output_capacity': 4000, 'fits': True, 'overflow_tokens': 0}

The example fits exactly. Reducing the input or the requested output creates room; increasing either one can make the request invalid before generation starts.

In [2]:
examples = [
    {'name': 'fits', 'input': 12_000, 'output': 4_000},
    {'name': 'overflows', 'input': 12_000, 'output': 5_000},
]
for example in examples:
    result = context_budget(16_000, example['input'], example['output'])
    print(example['name'], result)

fits {'remaining_output_capacity': 4000, 'fits': True, 'overflow_tokens': 0}
overflows {'remaining_output_capacity': 4000, 'fits': False, 'overflow_tokens': 1000}


## 2. What happens at overflow?

A request can fail with a hard error, be automatically truncated by a client, or be reduced by an application-managed policy. Never assume which behavior occurred: inspect the API and the final serialized request.

In [3]:
messages = [
    ('system', 'Follow the contract'),
    ('user', 'Original requirements: preserve exact IDs'),
    ('assistant', 'Earlier discussion'),
    ('user', 'Current task: produce a short answer'),
]

def keep_recent_messages(messages, max_messages):
    return messages[-max_messages:]

print('hard error:', 'reject request and report overflow')
print('keep recent:', keep_recent_messages(messages, 2))
print('preserve system:', [messages[0], *messages[-1:]])

hard error: reject request and report overflow
keep recent: [('assistant', 'Earlier discussion'), ('user', 'Current task: produce a short answer')]
preserve system: [('system', 'Follow the contract'), ('user', 'Current task: produce a short answer')]


The policies above have different failure modes. Keeping only recent messages can delete the original objective. Preserving the system message can still delete important user constraints. The policy must match the application's priorities.

## 3. Why the window is finite

In straightforward full attention, each token can interact with many other tokens. The number of pairwise relationships is approximately proportional to $n^2$. This is a scaling intuition, not a claim that every production implementation has exactly the same cost.

In [4]:
def relative_attention_work(sequence_length, baseline=1):
    return (sequence_length / baseline) ** 2

baseline = 4_000
for length in [4_000, 8_000, 16_000]:
    print(f'{length:>5,} tokens: {relative_attention_work(length, baseline):.0f}x baseline pairwise work')

4,000 tokens: 1x baseline pairwise work
8,000 tokens: 4x baseline pairwise work
16,000 tokens: 16x baseline pairwise work


During autoregressive generation, key/value caching avoids recomputing all earlier work for every new token, but the cache still grows with context length. A larger advertised window can therefore increase latency and memory use, and it does not guarantee equal quality for every distant detail.

## 4. Truncation versus compaction

Truncation removes information. Compaction replaces older information with a shorter representation. Both reduce token count, but only compaction attempts to preserve meaning. A summary is not equivalent to the original evidence, so authoritative data should remain outside the summary and be retrieved when exact wording matters.

In [5]:
conversation = ['goal: compare two plans', 'decision: choose plan B', 'constraint: budget is fixed', 'latest question: what is next?']

truncated = conversation[-2:]
compacted = ['summary: goal=compare plans; decision=plan B; constraint=budget fixed', conversation[-1]]

print('truncated:', truncated)
print('compacted:', compacted)

truncated: ['constraint: budget is fixed', 'latest question: what is next?']
compacted: ['summary: goal=compare plans; decision=plan B; constraint=budget fixed', 'latest question: what is next?']


A robust compaction record should separate stable facts, user preferences, decisions, constraints, unresolved work, completed work, and exact artifacts or identifiers that must not be paraphrased.

## 5. When a model appears to forget

Investigate the request before treating this as mysterious memory failure. The information may have never been sent, may have been truncated, may have been summarized imperfectly, may be buried or contradicted, or may be present but difficult for the model to use. A small output allowance can also prevent a complete answer.

Useful debugging questions:

1. What exact messages and documents were in the final request?
2. How many tokens did each section consume?
3. Which truncation or compaction policy ran?
4. Was the relevant fact preserved verbatim or only summarized?
5. Was it contradicted later in the prompt?
6. Did the model have enough output capacity?

## Context-management pattern

A practical pipeline is:

`collect state -> classify importance -> count tokens -> reserve output -> retrieve evidence -> compact or truncate -> validate -> send and log`

Prioritize system rules, the current task, safety constraints, active decisions, and directly relevant evidence. Remove redundant greetings, stale tool output, repeated instructions, and irrelevant history first.

## Worked example

A model has a 10,000-token context limit. The application assembles 1,000 system tokens, 6,500 history tokens, 2,000 retrieved-document tokens, 500 current-message tokens, and a 2,000-token output allowance. The total is 12,000, so the request exceeds the limit by 2,000 tokens.

In [6]:
parts = {
    'system': 1_000,
    'history': 6_500,
    'documents': 2_000,
    'current_message': 500,
    'requested_output': 2_000,
}
limit = 10_000
total = sum(parts.values())
print('total:', total)
print('overflow:', max(0, total - limit))
print('fits:', total <= limit)

total: 12000
overflow: 2000
fits: False


Possible remedies include removing low-value history, compacting history from 6,500 to 4,500 tokens, retrieving fewer documents, reducing the output ceiling, or rejecting the request clearly. Blindly removing the first 2,000 tokens may delete the original requirements.

## Self-check

- Given an input size and output allowance, can I determine whether a request fits?
- Can I distinguish an API hard error from client-side truncation?
- Can I explain why compaction is not the same as memory?
- Can I identify what was lost when a conversation forgot a requirement?
- Can I describe how context limits affect latency, cost, and reliability?